# Llama on Groq

[Llama3](https://github.com/meta-llama/llama3) is one of the most advanced open-source LLM models.

The original 80 billion parameter model (without quantanization) consumes about [160 Gb of VRAM](https://nlpcloud.com/how-to-install-and-deploy-llama-3-into-production.html). For practical use, especially for smaller organizations or individual developers, using a cloud service like Groq can be a more feasible option.

## Set up the environment and use Llama3

To set up and use Llama3 with Groq, you can follow these steps:

1. **Log in to Groq**: Visit [groq.com](https://groq.com) and log in with your credentials.

2. **Create an API Key**:
   - Navigate to the [API section](https://console.groq.com/keys) of the Groq console.
   - Generate a new API key. This key will be used to authenticate your requests to the Groq API.

3. **Set Up Your Environment**:
   - On your local machine, create a `.env` file. This file will store your API key securely.
   - Add the following line to your `.env` file:
     ```
     GROQ_API_KEY="PASTE_YOUR_API_KEY_HERE"
     ```
   - Replace `"PASTE_YOUR_API_KEY_HERE"` with the actual API key you generated.

4. **Consult the API Documentation**:
   - The API documentation, including details on usage, rate limits, and other configurations, can be found [here](https://console.groq.com/docs/quickstart). This documentation will guide you on how to interact with the API, make requests, and handle responses.

Install the Groq library:

In [1]:
!pip install groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.3/107.3 kB 542.1 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.9/84.9 kB 1.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.3/409.3 kB 1.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 3.8 MB/s eta 0:00:00a 0:00:01m


Install the dotenv library:

In [2]:
!pip install python-dotenv

Import the libraries:

In [1]:
# Import the standard libraries.
import os

# Import the third party libraries.
from dotenv import load_dotenv
from groq import Groq

Set the environment:

In [2]:
load_dotenv()

True

Test the API:

In [3]:
client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of fast language models in one sentence.",
        }
    ],
    model="llama3-70b-8192",
)

print(chat_completion.choices[0].message.content)

Fast language models are crucial because they enable real-time natural language understanding and generation, which is essential for applications such as chatbots, virtual assistants, and language translation, allowing for swift and efficient human-machine interaction.


## Summarize the whisper's transcript

Import the cleaned whishper transcript:

In [4]:
with open("./data/cleaned_whisper_transcript_max_context_64.txt", "r") as file:
    transcript = file.read()
transcript[0:1000]

"Welcome to IBM Think 2023. AI-generated art. AI-generated songs. AI, what is that? It sure is a lot of fun. But when foundation models are applied to big business, well, you need to think bigger. Because AI in business needs to be held to a higher standard. Built to be trusted, secured, and adaptable. This isn't simple automation that is only trained to do one thing. This is AI that is built and focused to work across your organization. This isn't committing to a single system. This is hybrid-ready AI that can scale across your systems. This isn't wondering where an answer came from. This is AI that can show its work. When you build AI into the core of your business, you can go so much further. This is more than AI. This is AI for business. Let's create. Please welcome Senior Vice President and Director of Research, IBM, Dr. Dario Gil. Hello. Welcome. Welcome. The last session of Think. And I understand some of you even had a drink. How special. So I hope you've enjoyed the last two d

<br>
Estimate a number of tokens in the transcript (1 token is ~4 characters for English):

In [5]:
len(transcript) / 4

6876.0

If we try to send a full text, we get an error message like this:
```bash
Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama3-70b-8192` in organization `org_` on tokens per minute (TPM): Limit 6000, Used 0, Requested 6897. Please try again in 8.97s. Visit https://console.groq.com/docs/rate-limits for more information.', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
```
This is because the rate limit for llama3-70b-8192 is [6000 per minute](https://console.groq.com/settings/limits) while we have almost 7000 tokens:

In [6]:
client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": f"Please summarize the following text in no less than 500 words:\n\n{transcript}",
        }
    ],
    model="llama3-70b-8192",
)

print(chat_completion.choices[0].message.content)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama3-70b-8192` in organization `org_01hxzrsnf8e94sgaavqxytxpxt` on tokens per minute (TPM): Limit 6000, Used 0, Requested 6897. Please try again in 8.97s. Visit https://console.groq.com/docs/rate-limits for more information.', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

The easiest way to fix it is to use a model with a higher rate limit, such as llama3-groq-70b-8192-tool-use-preview, which has a limit of [15,000 tokens per minute](https://console.groq.com/settings/limits).

Create the instruction:

In [7]:
instruction = (
    "Please divide the following text into several logical parts. "
    "The division should reflect a logical separation based on "
    "topics, themes, or any natural breakpoints in the text. "
    "Label each part with a brief heading that describes the focus or "
    "content of that section. Summarize every part. The summary every "
    "part has to be NOT less than 200 tokens.\n\n"
    "Please provide the output in the following format:\n\n"
    "Part 1: [Descriptive Heading]\n"
    "[The text from the first section]\n\n"
    "Part 2: [Descriptive Heading]\n"
    "[The text from the second section]\n"
)


In [8]:
client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": f"{instruction}\n\nThe text:\n\n{transcript}",
        }
    ],
    model="llama3-groq-70b-8192-tool-use-preview",
    temperature=0,
    max_tokens=8192   
)
print(chat_completion.choices[0].message.content)
print(f"\nA number of the completion tokens: {chat_completion.usage.completion_tokens} \n\n")

Part 1: Introduction to IBM Think 2023
The text begins with an introduction to IBM Think 2023, highlighting the excitement and potential of AI-generated art and songs. It emphasizes the importance of building AI into the core of businesses and becoming an AI value creator rather than just an AI user. The introduction also highlights the importance of community and the creativity of the open AI community.

Summary: The introduction to IBM Think 2023 emphasizes the potential of AI and the importance of becoming an AI value creator. It highlights the role of community and the creativity of the open AI community in driving innovation.

Part 2: Watson X Platform
The text then moves on to discuss Watson X, IBM's new integrated data and AI platform. Watson X consists of three primary parts: watsonx.data, watsonx.ai, and watsonx.governance. Watson X allows users to create and govern AI in real-time, and it is built on top of Red Hat OpenShift, providing seamless integration of Watson X compone

In general, it is quite a good result. We see a clear explanation for every section. However, the problem is that it is insensitive to the number of tokens. We asked for no less than 200 tokens for every section but received less than half that amount. This means we can lose important information in our summary. Additionally, this approach (using of a model with a larger token rate limit) has limitations for larger transcriptions.

## Alternative ways to summarize the transcript

There are a few strategies to address the issue of creating insufficiently detailed summaries and the constraints of a low token limit:
1. Chunk and Summarize (Map-Reduce Approach):
    * Divide the original transcript into multiple smaller chunks.
    * Summarize each chunk individually.
    * Concatenate these individual summaries (optionally).
    * Create a final summary from the concatenated summaries (optionally).      
&nbsp;
2. Incremental Summarization (Refine Approach):
    * Split the original transcript into smaller chunks.
    * Generate a summary for the first chunk.
    * Add the next chunk to the previous summary and generate a new summary.
    * Repeat this process until all chunks are summarized.   
&nbsp;
3. Utilize High-Capacity LLMs:
    * Use a language model with a very high token limit (e.g., 128k tokens).
    * This allows for processing larger portions of the transcript at once, enabling more comprehensive summaries.

&nbsp;
There is a good video related to this [topic](https://www.youtube.com/watch?v=f9_BWhCI4Zo).

### Chunk and Summarize (Map-Reduce Approach)

Renew the insruction:

In [9]:
instruction = (
    "I give a chunk of a full text. Please summarize this. "
    "You should divide the chunk into several logical parts. "
    "The division should reflect a logical separation based on "
    "topics, themes, or any natural breakpoints in the text. "
    "Label each part with a brief heading that describes the focus or "
    "content of that section. Summarize every part.\n\n"
    "Please provide the output in the following format:\n\n"
    "Part 1: [Descriptive Heading]\n"
    "[The text from the first section]\n\n"
    "Part 2: [Descriptive Heading]\n"
    "[The text from the second section]\n"
)

Divide the original transcript into 4 smaller chunks using [mr_summary_by_groq.py](./utils/mr_summary_by_groq.py) :

In [10]:
from utils.mr_summary_by_groq import split_transcript
num_chunks = 4
chunks = split_transcript(transcript, num_chunks)
for chunk in chunks:
    print(f"{chunk[:500]}...\n\n") 

Welcome to IBM Think 2023. AI-generated art. AI-generated songs. AI, what is that? It sure is a lot of fun. But when foundation models are applied to big business, well, you need to think bigger. Because AI in business needs to be held to a higher standard. Built to be trusted, secured, and adaptable. This isn't simple automation that is only trained to do one thing. This is AI that is built and focused to work across your organization. This isn't committing to a single system. This is hybrid-re...


Once the model passes all the thresholds across the benchmarks, it is packaged and marked as ready for use. For each model, we create a model card that lists all the details of the model. We will have many different models trained on different piles with different target goals. Next, we go to WatsonX.Governance to combine the data card that has the detailed provenance information for the data pile that was used for training with the model card that has the detailed information on how the m

Summarize the chunks:

In [11]:
from utils.mr_summary_by_groq import execute_tasks
responses = await execute_tasks(instruction, chunks)
token_num = 0
# Print each response message and a number of tokens of all
# response messages.
for num, response in enumerate(responses, start=1):
    print(f"Section #{num}\n\n{response.choices[0].message.content}\n\n")
    token_num += response.usage.completion_tokens
print(f"\nA number of the completion tokens: {token_num} \n\n")

Section #1

Part 1: Introduction to IBM Think 2023
Dr. Dario Gil, Senior Vice President and Director of Research at IBM, introduces the concept of Watson X, an integrated data and AI platform designed for value creators. Watson X consists of three primary parts: watsonx.data, watsonx.ai, and watsonx.governance. Watson X is built on top of Red Hat OpenShift, providing seamless integration and deployment of AI workloads in any IT environment.

Part 2: Watson X Platform Overview
Dr. Gil explains the Watson X platform's capabilities, including data preparation, model training, validation, and deployment. Watson X allows data scientists to access and manage data from various sources, filter and process it, and then use it to train models. The platform also provides tools for validation and deployment of the models, ensuring that they are ready for use in applications and solutions. Watson X is designed to work across hybrid cloud architectures and is built on top of Red Hat OpenShift, allow

We can see that this approach works, providing ~30-50% more tokens and a more detailed summary.

### Incremental Summarization (Refine Approach)

Split the original transcript into smaller chunks.
Generate a summary for the first chunk.
Add the next chunk to the previous summary and generate a new summary.
Repeat this process until all chunks are summarized.

In [12]:
instruction = (
    "I give a chunk of a full text. Please summarize this. "
    "You should divide the chunk into several logical parts. "
    "The division should reflect a logical separation based on "
    "topics, themes, or any natural breakpoints in the text. "
    "Label each part with a brief heading that describes the focus or "
    "content of that section. Summarize every part.\n\n"
    "Please provide the output in the following format:\n\n"
    "Part 1: [Descriptive Heading]\n"
    "[The text from the first section]\n\n"
    "Part 2: [Descriptive Heading]\n"
    "[The text from the second section]\n"
)

Divide the original transcript into 4 smaller chunks using [ref_summary_by_groq.py](./utils/ref_summary_by_groq.py) :

In [14]:
from utils.ref_summary_by_groq import split_transcript
num_chunks = 4
chunks = split_transcript(transcript, num_chunks)
for chunk in chunks:
    print(f"{chunk[:500]}...\n\n") 

Welcome to IBM Think 2023. AI-generated art. AI-generated songs. AI, what is that? It sure is a lot of fun. But when foundation models are applied to big business, well, you need to think bigger. Because AI in business needs to be held to a higher standard. Built to be trusted, secured, and adaptable. This isn't simple automation that is only trained to do one thing. This is AI that is built and focused to work across your organization. This isn't committing to a single system. This is hybrid-re...


Once the model passes all the thresholds across the benchmarks, it is packaged and marked as ready for use. For each model, we create a model card that lists all the details of the model. We will have many different models trained on different piles with different target goals. Next, we go to WatsonX.Governance to combine the data card that has the detailed provenance information for the data pile that was used for training with the model card that has the detailed information on how the m

Summarize the chunks:

In [21]:
from utils.ref_summary_by_groq import refine_summary
# Generate the refined summary using the refine_summary function.
final_summary, token_num = refine_summary(instruction, chunks)
# Output the final summary
print(f"\nFinal Summary:\n{final_summary}\n\n")
print(f"\nA number of the completion tokens: {token_num}\n\n")


Final Summary:
Part 1: Introduction to IBM Think 2023
Dr. Dario Gil introduces Watson X, an integrated data and AI platform designed for value creators. Watson X consists of three primary parts: watsonx.data, watsonx.ai, and watsonx.governance. Watson X is built on top of Red Hat OpenShift, providing seamless integration and deployment of AI workloads in any IT environment.

Part 2: Watson X Platform Overview
Dr. Gil explains the Watson X platform's capabilities, including data preparation, model training, validation, and deployment. Watson X allows data scientists to access and manage data from various sources, filter and process it, and then use it to train models. The platform also provides tools for validation and deployment of the models, ensuring that they are ready for use in applications and solutions. Watson X is designed to work across hybrid cloud architectures and is built on top of Red Hat OpenShift, allowing for easy deployment in any IT environment.

Part 3: Data and Mo

We can see that this approach works. It provides approximately 30-50% more tokens than the connection of separate chunk summaries. It has nearly the same number of tokens as the Map-Reduce approach, and the refined summary is written in a smoother manner. The disadvantage is that it takes more time because we need to make the API requests to Groq consecutively, not concurrently.

### Utilize High-Capacity LLMs:

#### The naive approach (copy and paste the transcript into ChatGPT-4o)

We send the following request:

```text
There is a transcript:
{HERE OUR TRANSCRIPT}

Please, do the following:

Please divide this transcript into several logical parts. The division should reflect a logical separation based on topics, themes, the speakers name, or any natural breakpoints in the text. Label each part with a brief heading that describes the focus or content of that section. Summarize every part. The summary every part has to be NOT less than 200 tokens.

Please provide the output in the following format:

    Part 1: [Descriptive Heading]    
    [The summary from the first section]
    
    Part 2: [Descriptive Heading]
    [The summary from the second section]

    Etc.
```

We get the response:

```markdown
### Part 1: **Introduction and Importance of AI in Business**

The event begins with a welcome to IBM Think 2023 and sets the stage by discussing the fun aspects of AI, such as AI-generated art and songs. However, the speaker quickly shifts focus to the more serious applications of AI in business, emphasizing the need for AI to meet higher standards of trust, security, and adaptability. The speaker highlights that AI in business is not about simple automation but about creating AI systems that can work across an organization, scale across systems, and provide transparent results. This introduction underscores the transformative potential of AI when integrated into the core of business operations, setting the tone for the discussions to follow.

### Part 2: **The Transformative Impact of AI Across Industries**

Dr. Dario Gil, Senior Vice President and Director of Research at IBM, takes the stage and reflects on the rapid advancements in AI over the past year. He expresses excitement about the possibilities AI will bring, noting its impact across various industries such as customer care, logistics, medicine, manufacturing, and energy. Dr. Gil acknowledges that while the pace of technological change can be daunting, the opportunities presented by foundation models and generative AI, when governed correctly, are immense. He advises businesses not to be mere users of AI but to become creators of AI value, emphasizing the importance of owning and controlling AI models and data to maximize their potential.

### Part 3: **Introduction to WatsonX and its Components**

Dr. Gil introduces WatsonX, IBM’s new integrated data and AI platform designed to empower businesses to become AI value creators. WatsonX is composed of three main parts: watsonx.data, a massive curated data repository; watsonx.ai, an enterprise studio for training, validating, and deploying AI models; and watsonx.governance, a set of tools to ensure responsible AI execution. He explains how these components work together seamlessly across the AI lifecycle, from data preparation and model training to validation, tuning, and deployment. Dr. Gil emphasizes that WatsonX is built on Red Hat OpenShift, enabling hybrid cloud integration and deployment of AI workloads across any IT environment, making it a versatile platform for enterprise AI.

### Part 4: **Detailed Workflow of AI Model Creation with WatsonX**

This section delves into the detailed workflow of creating AI models using WatsonX. Dr. Gil walks the audience through the process, starting with data preparation, where data scientists can access and filter data from various sources using watsonx.data. He explains the importance of curating data into "piles" based on different criteria, such as language or programming language, and filtering out irrelevant or harmful content. The workflow then moves to model training using watsonx.ai, where models are trained on IBM’s Vela AI supercomputer. Dr. Gil describes the process of tokenization, model validation, and the creation of model cards that document the training process. The section concludes with a discussion on tuning models for specific tasks and deploying them across different environments.

### Part 5: **WatsonX in Action and Its Value to Enterprises**

Dr. Gil showcases how WatsonX enables enterprises to deploy AI models effectively. He describes how businesses can use WatsonX to create and manage foundation models tailored to their specific needs, whether for customer care, scientific research, or industry-specific applications. He highlights several real-world examples, such as SAP using IBM Watson for digital assistants, Moderna applying foundation models for mRNA medicine prediction, and NASA using language models for climate research. This section underscores the practical applications of WatsonX across various industries and the tangible value it brings to businesses.

### Part 6: **Partnership with Hugging Face and the Power of Open AI Community**

The discussion shifts to IBM's partnership with Hugging Face, a leader in the open AI community. Dr. Gil invites Clem Delangue, co-founder and CEO of Hugging Face, to the stage. Clem shares the origins of Hugging Face and its evolution from a conversational AI company to a platform that hosts over 250,000 open models. He highlights the vibrant and rapidly growing open-source AI community, noting that 4,000 new models were added to Hugging Face in a single week. The partnership between IBM and Hugging Face aims to bring the creativity and diversity of the open AI community into the WatsonX platform, offering enterprises the best of both worlds: cutting-edge AI models from the community and the security and support of IBM.

### Part 7: **Future Outlook and Closing Remarks**

Dr. Gil concludes the event by reflecting on the significant advancements IBM has made in AI over the past year. He recaps the major announcements, including the launch of WatsonX, the creation of foundation models, and the development of IBM Vela, the AI supercomputer. He stresses the importance of having a business strategy for AI and offers key advice: act with urgency, be a value creator, bet on community, run AI efficiently across hybrid environments, and build responsibly with governance at the core. Dr. Gil’s closing remarks are a call to action for businesses to embrace AI fully, leveraging WatsonX to create and govern AI models that drive value and innovation.
```

As we can see, the ChatGPT 4o summary is more detailed, especially at the beginning, and it generally has a smoother manner. However, the Incremental Summarization is more detailed at the end and focuses on the general idea. Both variants can be used depending on the goal.

| #   | Incremental Summarization (Llama 3)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        | ChatGPT 4o                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           |
| --- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| 1   | Part 1: Introduction to IBM Think 2023<br>Dr. Dario Gil introduces Watson X, an integrated data and AI platform designed for value creators. Watson X consists of three primary parts: watsonx.data, watsonx.ai, and watsonx.governance. Watson X is built on top of Red Hat OpenShift, providing seamless integration and deployment of AI workloads in any IT environment.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               | Part 1: Introduction and Importance of AI in Business  <br>  <br>The event begins with a welcome to IBM Think 2023 and sets the stage by discussing the fun aspects of AI, such as AI-generated art and songs. However, the speaker quickly shifts focus to the more serious applications of AI in business, emphasizing the need for AI to meet higher standards of trust, security, and adaptability. The speaker highlights that AI in business is not about simple automation but about creating AI systems that can work across an organization, scale across systems, and provide transparent results. This introduction underscores the transformative potential of AI when integrated into the core of business operations, setting the tone for the discussions to follow.  <br>  <br>Part 2: The Transformative Impact of AI Across Industries  <br>  <br>Dr. Dario Gil, Senior Vice President and Director of Research at IBM, takes the stage and reflects on the rapid advancements in AI over the past year. He expresses excitement about the possibilities AI will bring, noting its impact across various industries such as customer care, logistics, medicine, manufacturing, and energy. Dr. Gil acknowledges that while the pace of technological change can be daunting, the opportunities presented by foundation models and generative AI, when governed correctly, are immense. He advises businesses not to be mere users of AI but to become creators of AI value, emphasizing the importance of owning and controlling AI models and data to maximize their potential.                                                   |
| 2   | Part 2: Watson X Platform Overview<br>Dr. Gil explains the Watson X platform's capabilities, including data preparation, model training, validation, and deployment. Watson X allows data scientists to access and manage data from various sources, filter and process it, and then use it to train models. The platform also provides tools for validation and deployment of the models, ensuring that they are ready for use in applications and solutions. Watson X is designed to work across hybrid cloud architectures and is built on top of Red Hat OpenShift, allowing for easy deployment in any IT environment.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                | Part 3: Introduction to WatsonX and its Components  <br>   <br>Dr. Gil introduces WatsonX, IBM’s new integrated data and AI platform designed to empower businesses to become AI value creators. WatsonX is composed of three main parts: watsonx.data, a massive curated data repository; watsonx.ai, an enterprise studio for training, validating, and deploying AI models; and watsonx.governance, a set of tools to ensure responsible AI execution. He explains how these components work together seamlessly across the AI lifecycle, from data preparation and model training to validation, tuning, and deployment. Dr. Gil emphasizes that WatsonX is built on Red Hat OpenShift, enabling hybrid cloud integration and deployment of AI workloads across any IT environment, making it a versatile platform for enterprise AI.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
| 3   | Part 3: Data and Model Factory<br>Dr. Gil discusses the concept of the data and model factory, which continuously monitors and updates models in both .data and .ai. This process is central to solid and proper governance of AI. Watson X has already produced over 20 foundation models across modalities like language, code, geospatial, and chemistry, and spanning different sizes of models -- from hundreds of millions to billions of parameters. These models have been infused into IBM products, Red Hat products, and/or partners products.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  | Part 4: Detailed Workflow of AI Model Creation with WatsonX  <br>  <br>This section delves into the detailed workflow of creating AI models using WatsonX. Dr. Gil walks the audience through the process, starting with data preparation, where data scientists can access and filter data from various sources using watsonx.data. He explains the importance of curating data into "piles" based on different criteria, such as language or programming language, and filtering out irrelevant or harmful content. The workflow then moves to model training using watsonx.ai, where models are trained on IBM’s Vela AI supercomputer. Dr. Gil describes the process of tokenization, model validation, and the creation of model cards that document the training process. The section concludes with a discussion on tuning models for specific tasks and deploying them across different environments.  <br>  <br>Part 5: WatsonX in Action and Its Value to Enterprises  <br>  <br>Dr. Gil showcases how WatsonX enables enterprises to deploy AI models effectively. He describes how businesses can use WatsonX to create and manage foundation models tailored to their specific needs, whether for customer care, scientific research, or industry-specific applications. He highlights several real-world examples, such as SAP using IBM Watson for digital assistants, Moderna applying foundation models for mRNA medicine prediction, and NASA using language models for climate research. This section underscores the practical applications of WatsonX across various industries and the tangible value it brings to businesses. |
| 4   | Part 4: Partnership with Hugging Face<br>Dr. Gil announces a new partnership with Hugging Face, a company that specializes in open source AI. Hugging Face co-founder and CEO, Claym DeLong, shares that since the release of ChatGPT, 100,000 new models have been added on HuggingFace. Companies don't train models just to train models, as it costs money to train models. But the truth is, if you look at how AI is built, when you can build smaller, more specialized customized models for your use cases, they end up being cheaper, they end up being more efficient, and they end up being better for your use case, right? Just the same way every single technology company learned how to write code, and to have a different code base than their competitors or than companies in other fields, we're seeing the same thing for AI, right? Every single company needs... Every single company needs to train their own models, optimize their own models, learn how to run these models at scale. Every single company needs to build their own chat GPT because if they don't, they won't be able to differentiate. They won't be able to create the unique technology value that they've been building for their customers and they'll lose control, right, if they start outsourcing it. So that's what we're seeing on Hugin. Face and in the ecosystem as a whole. It's back to this philosophy of don't just be a prompt tuner user, right, be a value creator with all of this.                   | Part 6: Partnership with Hugging Face and the Power of Open AI Community  <br>  <br>The discussion shifts to IBM's partnership with Hugging Face, a leader in the open AI community. Dr. Gil invites Clem Delangue, co-founder and CEO of Hugging Face, to the stage. Clem shares the origins of Hugging Face and its evolution from a conversational AI company to a platform that hosts over 250,000 open models. He highlights the vibrant and rapidly growing open-source AI community, noting that 4,000 new models were added to Hugging Face in a single week. The partnership between IBM and Hugging Face aims to bring the creativity and diversity of the open AI community into the WatsonX platform, offering enterprises the best of both worlds: cutting-edge AI models from the community and the security and support of IBM.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       |
| 5   | Part 5: IBM's Achievements and Future Plans<br>Dr. Gil summarizes the achievements of IBM in the past year, including the announcement of WatsonX, a comprehensive platform that allows you to create and govern AI in real time. IBM has also announced a set of foundation models, a data model factory, and important collaborations to advance AI and bring it to the enterprise. They have also launched the IBM Vela or cloud native AI supercomputer to train foundation models with bare metal performance while giving us the flexibility of the cloud. And they announced that they're making it available as a service. Last year, they launched the Telum in C 16. It's an engineering marvel, and IBM's first processor to have on-chip accelerator for AI inferencing. It can process 300 billion inferences per day with one millisecond latencies. This means that now you can infuse AI into every transaction in C 16 for applications like fraud detection and others in real time. Using the same core architecture as Telum, they built the IBM Research AIU. Which is optimized to give superior performance for foundation models and enable with the Red Hat software stack. And at IBM Research, they are incubating powerful AIU systems designed and optimized for enterprise AI inference and tuning. So a truly fantastic year. And this is just the start of all the amazing things that they're building and developing for you and that they will be sharing with you in the coming years. | Part 7: Future Outlook and Closing Remarks  <br>  <br>Dr. Gil concludes the event by reflecting on the significant advancements IBM has made in AI over the past year. He recaps the major announcements, including the launch of WatsonX, the creation of foundation models, and the development of IBM Vela, the AI supercomputer. He stresses the importance of having a business strategy for AI and offers key advice: act with urgency, be a value creator, bet on community, run AI efficiently across hybrid environments, and build responsibly with governance at the core. Dr. Gil’s closing remarks are a call to action for businesses to embrace AI fully, leveraging WatsonX to create and govern AI models that drive value and innovation.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          |
| 6   | Part 6: Importance of Responsible AI<br>Dr. Gil emphasizes the importance of building responsibly, transparently, and putting governance into the heart of your AI lifecycle. Continuously govern the data you use and the AI you deploy. And co-create with trusted partners. Trust is your ultimate license to operate. With your AI business strategy against these recommendations, you will be in a prime position to do amazing things with foundation models and generative AI. They have built WatsonX so that you can do just that. And I hope you join us, because we cannot wait to get started on this journey with you. Thank you.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      |